# 예제 franka_ex15: FR3 IK 기반 텔레오퍼레이션 (notebook 버전)

`ex11` (Servo) 과 비슷한 텔레오퍼레이션이지만, **Servo 노드 없이** `compute_ik` 서비스 +
`fr3_arm_controller` 직접 발행으로 구현. Servo 의 특이점 차단이 부담스러울 때 좋다.
원본 6-DOF `ex15_keyboard_ik.py` 가 curses 기반이라, 노트북에서는 셀에서 명시적으로
스텝을 호출하는 형태로 옮긴다.

**6-DOF 예제와 다른 점**
- 컨트롤러 토픽 `/fr3_arm_controller/joint_trajectory`
- 조인트 7개 (`fr3_joint1..7`)
- 끝단 링크 `fr3_hand_tcp`, planning group `fr3_arm`, frame `fr3_link0`
- 7-DOF redundancy 덕에 같은 변위에서도 IK 해를 더 자주 찾는다
- `home` 없음 → 시작/복귀는 `ready`

**학습 내용**
- TF lookup 으로 현재 EE 절대 자세 받아 → 작은 변위 더해 → `compute_ik` → `JointTrajectory` 발행
- Servo 없는 텔레오퍼레이션의 단순함 / 책임 (특이점 직접 회피해야 함)
- 누적 끝단 경로 LINE_STRIP 시각화

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/ik_teleop_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 상수

In [ ]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC = '/ik_teleop_markers'
ARM_TRAJ_TOPIC = '/fr3_arm_controller/joint_trajectory'

LINEAR_STEP  = 0.005    # 한 step 당 5mm 이동 (기본)
ANGULAR_STEP = 0.05     # 한 step 당 ~2.9° 회전 (기본)
STEP_RATE_HZ = 30.0
DEFAULT_STEPS = 20      # 한 명령 당 기본 스텝 수 (~0.66s)

## 2. 초기화

In [ ]:
import rclpy, math, time
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup
from moveit_msgs.srv import GetPositionIK
from moveit_msgs.msg import RobotState, MoveItErrorCodes
from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
from visualization_msgs.msg import MarkerArray, Marker
from std_msgs.msg import ColorRGBA
from geometry_msgs.msg import Pose, Point, Quaternion
from builtin_interfaces.msg import Duration
import tf_transformations
import tf2_ros

In [ ]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

In [ ]:
node = Node(
    'franka_ex15_ik_teleop_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client = ActionClient(node, MoveGroup, 'move_action')

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex15 노트북 노드 생성 완료 ===')
marker_pub = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)
ik_client = node.create_client(GetPositionIK, 'compute_ik')
traj_pub = node.create_publisher(JointTrajectory, ARM_TRAJ_TOPIC, 10)
tf_buffer = tf2_ros.Buffer()
tf_listener = tf2_ros.TransformListener(tf_buffer, node)

## 3. 서버 준비

In [ ]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + /joint_states 준비됨')

wait_for_ready()

## 4. SRDF `ready` 자세

In [ ]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

## 5. Pose / MoveGroup 헬퍼

In [ ]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

In [ ]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only, replan=not plan_only, replan_attempts=3 if not plan_only else 0)
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

def go_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

def plan_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                      planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

## 6. 현재 EE pose 읽기 + IK / 스트리밍 헬퍼

In [ ]:
def get_current_joints() -> dict:
    if joint_state['msg'] is None:
        return {}
    msg = joint_state['msg']
    return {n: p for n, p in zip(msg.name, msg.position) if n in ARM_JOINTS}

def get_current_pose():
    try:
        tr = tf_buffer.lookup_transform(REFERENCE_FRAME, END_EFFECTOR_LINK, rclpy.time.Time())
    except Exception:
        return None
    p = Pose()
    p.position = Point(x=tr.transform.translation.x,
                       y=tr.transform.translation.y,
                       z=tr.transform.translation.z)
    p.orientation = Quaternion(x=tr.transform.rotation.x,
                               y=tr.transform.rotation.y,
                               z=tr.transform.rotation.z,
                               w=tr.transform.rotation.w)
    return p

def compute_ik(target_pose: Pose, timeout_sec: float = 0.05):
    req = GetPositionIK.Request()
    req.ik_request.group_name = PLANNING_GROUP
    req.ik_request.pose_stamped.header.frame_id = REFERENCE_FRAME
    req.ik_request.pose_stamped.pose = target_pose
    req.ik_request.timeout.sec = 0
    req.ik_request.timeout.nanosec = int(timeout_sec * 1e9)
    cur = get_current_joints()
    if cur:
        rs = RobotState()
        rs.joint_state.name = list(cur.keys())
        rs.joint_state.position = list(cur.values())
        req.ik_request.robot_state = rs
    fut = ik_client.call_async(req)
    deadline = time.time() + 0.1
    while not fut.done() and time.time() < deadline:
        rclpy.spin_once(node, timeout_sec=0.005)
    if fut.done() and fut.result() is not None:
        res = fut.result()
        if res.error_code.val == MoveItErrorCodes.SUCCESS:
            return {n: p for n, p in zip(res.solution.joint_state.name,
                                          res.solution.joint_state.position)
                    if n in ARM_JOINTS}
    return None

def publish_joint_cmd(jvals: dict, duration_s: float = 0.05):
    msg = JointTrajectory()
    msg.header.stamp = node.get_clock().now().to_msg()
    msg.joint_names = ARM_JOINTS
    pt = JointTrajectoryPoint()
    pt.positions = [jvals.get(n, 0.0) for n in ARM_JOINTS]
    sec = int(duration_s)
    pt.time_from_start = Duration(sec=sec, nanosec=int((duration_s - sec) * 1e9))
    msg.points.append(pt)
    traj_pub.publish(msg)

## 7. 텔레오퍼레이션 step 함수

`step(linear=(dx, dy, dz), angular=(droll, dpitch, dyaw), steps=N)` 한 번 호출 =
30Hz 로 N 번 반복하며 매 주기 5mm/2.9° 정도씩 누적 변위를 적용한다.
RViz 의 `/ik_teleop_markers` 에 끝단 누적 경로가 LINE_STRIP 으로 표시된다.

In [ ]:
COLOR_PATH    = ColorRGBA(r=0.2, g=1.0, b=0.4, a=0.85)
ee_path = []

def publish_path():
    if len(ee_path) < 2:
        return
    stamp = node.get_clock().now().to_msg()
    line = Marker()
    line.header.frame_id = REFERENCE_FRAME
    line.header.stamp = stamp
    line.ns = 'ee_path'
    line.id = 0
    line.type = Marker.LINE_STRIP
    line.action = Marker.ADD
    line.pose.orientation.w = 1.0
    line.scale.x = 0.005
    line.color = COLOR_PATH
    line.points = [Point(x=p[0], y=p[1], z=p[2]) for p in ee_path]
    marker_pub.publish(MarkerArray(markers=[line]))

def step(linear=(0.0, 0.0, 0.0), angular=(0.0, 0.0, 0.0),
         steps: int = DEFAULT_STEPS):
    '''현재 EE pose 에 매 주기 작은 변위를 더해 IK 풀고 컨트롤러로 발행.
    linear / angular 단위벡터, 크기는 LINEAR_STEP / ANGULAR_STEP 으로 곱한다.'''
    dt = 1.0 / STEP_RATE_HZ
    fail = 0
    for _ in range(steps):
        loop_t = time.time()
        cur = get_current_pose()
        if cur is None:
            time.sleep(dt)
            continue
        # 다음 목표 pose
        target = Pose()
        target.position = Point(
            x=cur.position.x + linear[0] * LINEAR_STEP,
            y=cur.position.y + linear[1] * LINEAR_STEP,
            z=cur.position.z + linear[2] * LINEAR_STEP,
        )
        q_cur = [cur.orientation.x, cur.orientation.y, cur.orientation.z, cur.orientation.w]
        q_delta = tf_transformations.quaternion_from_euler(
            angular[0] * ANGULAR_STEP, angular[1] * ANGULAR_STEP, angular[2] * ANGULAR_STEP)
        q_new = tf_transformations.quaternion_multiply(q_cur, q_delta)
        target.orientation = Quaternion(x=q_new[0], y=q_new[1], z=q_new[2], w=q_new[3])

        sol = compute_ik(target, timeout_sec=0.04)
        if sol:
            publish_joint_cmd(sol, duration_s=dt * 1.5)
            ee_path.append((target.position.x, target.position.y, target.position.z))
            fail = 0
        else:
            fail += 1
            if fail % 5 == 1:
                node.get_logger().warn('IK 실패 (작업영역/특이점 한계)')
        # 주기
        elapsed = time.time() - loop_t
        if dt - elapsed > 0:
            time.sleep(dt - elapsed)
    publish_path()

## 8. ready 자세 + 시작 위치

In [ ]:
go_to_joint_goal(ready_target, vel=0.4)
time.sleep(1.0)

# 텔레오퍼레이션 시작 자세: base 앞 ~50cm, 60cm 높이, 그리퍼 아래
start_pose = make_pose(0.50, 0.00, 0.50, math.pi, 0.0, 0.0)
go_to_pose_goal(start_pose, vel=0.3)
time.sleep(1.0)
ee_path.clear()  # 누적 경로 초기화

## 9. 시연 셀 — 셀 한 번 = 한 방향으로 짧게 이동

각 셀은 ~0.66초 동안 한 방향으로 끝단을 슬라이드한다.
RViz 의 끝단 누적 경로가 시안색 LINE_STRIP 으로 갱신된다.

### 9-1. 전진 +X

In [ ]:
step(linear=(1.0, 0.0, 0.0), steps=20)

### 9-2. 좌측 +Y

In [ ]:
step(linear=(0.0, 1.0, 0.0), steps=20)

### 9-3. 상승 +Z

In [ ]:
step(linear=(0.0, 0.0, 1.0), steps=20)

### 9-4. 후진 -X

In [ ]:
step(linear=(-1.0, 0.0, 0.0), steps=20)

### 9-5. 우측 -Y

In [ ]:
step(linear=(0.0, -1.0, 0.0), steps=20)

### 9-6. 하강 -Z

In [ ]:
step(linear=(0.0, 0.0, -1.0), steps=20)

### 9-7. yaw 회전 +Z

In [ ]:
step(angular=(0.0, 0.0, 1.0), steps=20)

### 9-8. 자유 명령 — 셀에 직접 값 넣어 시도

linear/angular 에 -1.0 ~ 1.0 단위벡터, steps 로 스텝 수 조절.

In [ ]:
# 예: 우측 + 상승 동시에 30스텝 (~1초)
step(linear=(0.0, -0.7, 0.7), steps=30)

## 10. ready 복귀 (MoveGroup)

In [ ]:
go_to_joint_goal(ready_target, vel=0.4)
node.get_logger().info('=== franka_ex15 완료! ===')

## 11. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass